# Original chest X-ray CNN: compact benchmark report

This is the streamlined companion to `04_chest_Xray_CNN.ipynb`. It uses the already trained `cnn_pneumonia_model.pt`, evaluates the same 624-image folder that the original notebook used for validation, and reproduces the presentation figures with `make_cnn_presentation.py`.

The validation folder was used for early stopping/model selection, so these values describe the saved notebook benchmark rather than performance on an untouched clinical test set. Patient independence is also unverified.

## Model and data

The input is one grayscale channel at 224 × 224 pixels. The network contains two convolution/pooling stages followed by three linear layers:

```text
1×224×224
  → Conv2d(1, 2, kernel=16, stride=4) → ReLU → MaxPool2d(2)
  → Conv2d(2, 4, kernel=5)            → ReLU → MaxPool2d(2)
  → Flatten(484) → Linear(484,16) → Linear(16,8) → Linear(8,1) → Sigmoid
```

`NORMAL` is class 0 and `PNEUMONIA` is class 1. A probability greater than 0.5 is classified as pneumonia. HiResCAM explains the predicted-class pre-sigmoid score at the last convolutional layer, `conv[3]`, whose native map is 22 × 22.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

ROOT = Path.cwd()
# When opened directly in Colab, clone the repository once so the checkpoint, data,
# plotting script and linked panel outputs are available to the notebook.
if not (ROOT / 'make_cnn_presentation.py').is_file() and Path('/content').is_dir():
    colab_root = Path('/content/xray_project')
    if not (colab_root / 'make_cnn_presentation.py').is_file():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/wxchew/xray_project.git', str(colab_root)
        ], check=True)
    ROOT = colab_root
CHECKPOINT = ROOT / 'cnn_pneumonia_model.pt'
DATA_DIR = ROOT / 'data/chest_xray_224/test'
OUTPUT_DIR = ROOT / 'cnn_presentation'
PLOT_SCRIPT = ROOT / 'make_cnn_presentation.py'
EXPECTED_SHA256 = 'eff4cb1b143731f997150d391744f2b51f03285b6ea6282baabad9542f1a399f'

assert CHECKPOINT.is_file(), f'Missing {CHECKPOINT}'
assert PLOT_SCRIPT.is_file(), f'Missing {PLOT_SCRIPT}'
assert len(list(DATA_DIR.glob('*/*.jpeg'))) == 624, 'Expected 624 validation images'
assert hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() == EXPECTED_SHA256, (
    'Checkpoint differs from the reviewed original CNN benchmark'
)
print('Checkpoint and 624-image validation set are ready.')

## Reproduce the evaluation

The following cell performs inference, calculates the confusion matrix and ranking metrics, computes 2,000 image-bootstrap confidence intervals with seed 0, selects deterministic TP/TN/FP/FN examples, calculates HiResCAM, and exports PNG, PDF, CSV and JSON results. It does not retrain or modify the checkpoint.

In [ ]:
subprocess.run([
    sys.executable, str(PLOT_SCRIPT),
    '--root', str(ROOT),
    '--checkpoint', CHECKPOINT.name,
    '--data-dir', str(DATA_DIR.relative_to(ROOT)),
    '--output-dir', str(OUTPUT_DIR.relative_to(ROOT)),
    '--threshold', '0.5',
    '--bootstrap', '2000',
    '--seed', '0',
], check=True)

In [ ]:
results = json.loads((OUTPUT_DIR / 'cnn_results.json').read_text())
print('Confusion matrix [[TN, FP], [FN, TP]]:', results['confusion_matrix']['values'])
for key in ('accuracy', 'sensitivity', 'specificity', 'precision', 'f1',
            'balanced_accuracy', 'roc_auc', 'average_precision'):
    item = results['metrics'][key]
    low, high = item['ci_95_percentile']
    print(f"{key:>18}: {item['estimate']:.4f} (95% bootstrap CI {low:.4f}–{high:.4f})")

## Result

| Metric | Estimate | 95% image-bootstrap interval |
|---|---:|---:|
| Accuracy | 0.827 | 0.798–0.857 |
| Sensitivity | 0.977 | 0.961–0.990 |
| Specificity | 0.577 | 0.513–0.639 |
| Precision | 0.794 | 0.758–0.832 |
| F1 score | 0.876 | 0.852–0.899 |
| Balanced accuracy | 0.777 | 0.744–0.809 |
| ROC-AUC | 0.910 | 0.885–0.933 |
| Average precision | 0.937 | 0.914–0.957 |

The benchmark detects most pneumonia images (381 of 390) but incorrectly labels 99 of 234 normal images as pneumonia. The high sensitivity therefore comes with modest specificity. ROC-AUC uses all continuous probabilities and is independent of the displayed 0.5 decision threshold. HiResCAM is an explanation of model evidence at coarse spatial resolution; it is not a pneumonia lesion segmentation.

## A — Confusion matrix

The next cell displays the generated PNG and reports the corresponding PDF path.

In [ ]:
from IPython.display import Image as NotebookImage, display
display(NotebookImage(filename=str(OUTPUT_DIR / 'panel_a_confusion_matrix.png')))
print('PDF:', OUTPUT_DIR / 'panel_a_confusion_matrix.pdf')

## B — ROC curve

The curve uses continuous pneumonia probabilities across all thresholds.

In [ ]:
display(NotebookImage(filename=str(OUTPUT_DIR / 'panel_b_roc_curve.png')))
print('PDF:', OUTPUT_DIR / 'panel_b_roc_curve.pdf')

## D — Deterministic HiResCAM examples

Each column shows one example closest to the median prediction confidence within its outcome category. Several highlights fall on borders or outside the lung fields, which is evidence that the classifier may use shortcuts.

In [ ]:
display(NotebookImage(filename=str(OUTPUT_DIR / 'panel_d_hirescam_examples.png')))
print('PDF:', OUTPUT_DIR / 'panel_d_hirescam_examples.pdf')